# Notebook for demonstrating and testing MAS capabilities of data insertion

The use of individual workflows of this system is also available to users and developers. 

This magazine will demonstrate how to use the workflow to add data to the data lake by user request.

Also here we will test the capabilities of the presented workflow in terms of the accuracy of autofilling metadata properties.

## 0. Generating some tested datalake for working with.

In [ ]:
import sys
sys.path.append('./../src')

In [ ]:
import os
import shutil
import json
from data_management.local_datalake_management import LocalDataLake

if os.path.exists('./add_testing_storage'):
    shutil.rmtree('./add_testing_storage')

with open('raw_testing_material/imgs_metadata_rich_descr.json', 'r') as i_stream:
    some_dict = json.load(i_stream)


local_datalake_instance = LocalDataLake.create('add_testing_storage', './')

local_datalake_instance.create_dataset(
    'fluorescent_microscopy', 
    '3d fluorescent images of neurons, captured by confocal microscope', 
    some_dict
    )

local_datalake_instance.create_dataset(
    'dendritic_spikes_9009', 
    'Dataset of 3D meshes of some dendritic spines.', 
    some_dict
    )

local_datalake_instance.create_dataset(
    'mice_behavioral', 
    'Videos of mice with different conditions for analysing behavioral of healthy and ill mice.', 
    some_dict
    )

local_datalake_instance.create_dataset(
    'neurons_activity', 
    'Dataset of generated neurons activities time series, captured from video.', 
    some_dict
    )

In [ ]:
TEST_TASK_1 = "Hi! Add neuron images from the directory './raw_testing_material/imgs/' to my dataset fluorescent images dataset. To describe images use metadata specified in the file './raw_testing_material/imgs_meta.xml'"

## 1. Data insertion demonstration in data lake with MAS

Adding data to data lake is implemented using ReACT workflow type. Here is an example of such usage.

In [ ]:
import os
import yaml

prompts_names = ['adding_data_react_auto_sp.yaml']

with open(os.path.join('../src/prompts_templates/adding_data_to_dataset', prompts_names[0])) as stream:
    react_add_data_prompt = yaml.safe_load(stream)['system_prompt']

react_add_data_prompt

The function for async testing of data insertion presented below. It takes as an input some testing parameters (system propmpt, user's task) and add info into datalake.

In [ ]:
import concurrent.futures
from time import time

from scidatamas.add_data_workflow import (AddDataToDatasetFlow,
                                          AddDataWorkflowState)
from utils import TokenUsageCallbackCounter


def add_data(user_task: str, datalake:LocalDataLake, system_prompt: str, model: str, provider: str):
    add_data_workflow = AddDataToDatasetFlow(system_prompt=system_prompt, datalake=datalake, model=model, provider=provider, is_auto=True)

    timeout = 300

    working_state = AddDataWorkflowState()
    working_state['users_task'] = user_task
    wf = add_data_workflow.get_workflow()
    wf = wf.compile()

    errs_cnt = 0
    while errs_cnt <= 5:
        start = time()
        with concurrent.futures.ThreadPoolExecutor() as executor:
            handler = TokenUsageCallbackCounter()
            config = {"callbacks":[handler]}
            future = executor.submit(wf.invoke, working_state, config=config)
            try:
                working_state = future.result(timeout=timeout)
            except concurrent.futures.TimeoutError:
                print(f"Function timed out after {timeout} seconds")
                errs_cnt += 1
                continue
            except Exception as exp:
                print(f"Error: {str(exp)}. Going to restart")
                errs_cnt += 1
                continue
        timing = float(time() - start)
        break

    if errs_cnt > 5:
        timing = None
            
    total_tokens = handler.total_tokens
    return working_state, total_tokens, timing, errs_cnt

Here is example of invokation.

In [ ]:
final_state, total_tokens, timing, errs_cnt = add_data(TEST_TASK_1, local_datalake_instance, react_add_data_prompt, model='mistral-large-latest', provider='mistralai')

And here is results of working

In [ ]:
final_state['currently_filled_schema'], total_tokens, timing, errs_cnt

## 2. Testing metadata filling accuracy - to main result

In [ ]:
from typing import Dict, Any

TRUE_DICT = {
    'camera_pixel_size_um': 6.45,
    'binning': 2,
    'magnification': 60,
    'number_of_pixels_x': 2048,
    'number_of_pixels_y': 2048,
    'number_of_pixels_z': 25,
    'pixel_size_x': 0.215,
    'pixel_size_y': 0.215,
    'pixel_size_z': 1,
    'physical_size_x': 440.32,
    'physical_size_y': 440.32,
    'physical_size_z': 25,
}
EPS = 10e-8

def validating_res(filled_schema: Dict[str, Any], true_scema: Dict[str, Any], eps: float):
    bad_filled = len(list(set(filled_schema.keys()) - set(true_scema.keys()))) + len(list(set(true_scema.keys()) - set(filled_schema.keys())))
    good_filled = 0

    for key, value in true_scema.items():
        if key not in filled_schema.keys():
            continue
        else:
            filled_value = float(filled_schema[key])
            if abs(filled_value - value) < eps:
                good_filled += 1
            else:
                bad_filled += 1
    
    return good_filled, bad_filled


In [ ]:
MODELS = {'mistralai':'mistral-large-latest'}
LAUNCHES_CNT = 16
TASK = "Hi! Add neuron images from the directory './raw_testing_material/imgs/' to my dataset fluorescent images dataset. To describe images use metadata specified in the file './raw_testing_material/imgs_meta.xml'"

results = {}

for model_name, model in MODELS.items():
    stats = []
    for i in range(LAUNCHES_CNT):
        print(f'Currently runs: \'{model}\' model, launch #{i+1}...')
        gen_res, tokens, cur_time, errs_cnt = add_data(TASK, local_datalake_instance, react_add_data_prompt, model=model, provider=model_name)
        good_filled, bad_filled = validating_res(gen_res['currently_filled_schema'], TRUE_DICT, EPS)
        print(f'Good filled fields: {good_filled}, bad filled fields: {bad_filled}, tokens: {tokens}, time: {cur_time}, errors during execution: {errs_cnt}.')            
        stats.append((good_filled, bad_filled, tokens, cur_time, errs_cnt))
        
    results.update({model_name: stats})
        

In [ ]:
import numpy as np

for model_name, model_stats in results.items():
    good_filled = [run_stat[0] for run_stat in model_stats]
    bad_filled = [run_stat[1] for run_stat in model_stats]
    tokens = [run_stat[2] for run_stat in model_stats]
    times = [run_stat[3] for run_stat in model_stats]
    
    mean_good, std_good = np.mean(good_filled), np.std(good_filled)
    mean_bad, std_bad = np.mean(bad_filled), np.std(bad_filled)
    mean_tokens, std_tokens = np.mean(tokens), np.std(tokens)
    mean_times, std_times = np.mean(times), np.std(times)

    print(f"--- '{model_name}' INFO ---")
    print(f"CORRECT FILLED:   mean={mean_good:6.6},\tstd={std_good:6.6}")
    print(f"INCORRECT FILLED: mean={mean_bad:6.6},\tstd={std_bad:6.6}")
    print(f"TOKENS:         mean={mean_tokens:6.6},\tstd={std_tokens:6.6}")
    print(f"TIME:           mean={mean_times:6.6},\tstd={std_times:6.6}")
    print()

## 3. Testing metadata filling (based on true values) WITH more less described fields! (like without formulas and units) - to suppl result

In [ ]:
import shutil
import json
from data_management.local_datalake_management import LocalDataLake, LocalRichDataset, DatasetInfo

if os.path.exists('./add_testing_storage'):
    shutil.rmtree('./add_testing_storage')

with open('raw_testing_material/imgs_metadata_poor_descr.json', 'r') as i_stream:
    some_dict = json.load(i_stream)


local_datalake_instance = LocalDataLake.create('add_testing_storage', './')

local_datalake_instance.create_dataset(
    'fluorescent_microscopy', 
    '3d fluorescent images of neurons, captured by confocal microscope', 
    some_dict
    )

local_datalake_instance.create_dataset(
    'dendritic_spikes_9009', 
    'Dataset of 3D meshes of some dendritic spines.', 
    some_dict
    )

local_datalake_instance.create_dataset(
    'mice_behavioral', 
    'Videos of mice with different conditions for analysing behavioral of healthy and ill mice.', 
    some_dict
    )

local_datalake_instance.create_dataset(
    'neurons_activity', 
    'Dataset of generated neurons activities time series, captured from video.', 
    some_dict
    )

In [ ]:
MODELS = {'mistralai':'mistral-large-latest'}
LAUNCHES_CNT = 16
TASK = "Hi! Add neuron images from the directory './raw_testing_material/imgs/' to my dataset fluorescent images dataset. To describe images use metadata specified in the file './raw_testing_material/imgs_meta.xml'"

results = {}

for model_name, model in MODELS.items():
    stats = []
    for i in range(LAUNCHES_CNT):
        print(f'Currently runs: \'{model}\' model, launch #{i+1}...')
        gen_res, tokens, cur_time, errs_cnt = add_data(TASK, local_datalake_instance, react_add_data_prompt, model=model, provider=model_name)
        good_filled, bad_filled = validating_res(gen_res['currently_filled_schema'], TRUE_DICT, EPS)
        print(f'Good filled fields: {good_filled}, bad filled fields: {bad_filled}, tokens: {tokens}, time: {cur_time}, errors during execution: {errs_cnt}.')            
        stats.append((good_filled, bad_filled, tokens, cur_time, errs_cnt))
        
    results.update({model_name: stats})
        

In [ ]:
for model_name, model_stats in results.items():
    good_filled = [run_stat[0] for run_stat in model_stats]
    bad_filled = [run_stat[1] for run_stat in model_stats]
    tokens = [run_stat[2] for run_stat in model_stats]
    times = [run_stat[3] for run_stat in model_stats]
    
    mean_good, std_good = np.mean(good_filled), np.std(good_filled)
    mean_bad, std_bad = np.mean(bad_filled), np.std(bad_filled)
    mean_tokens, std_tokens = np.mean(tokens), np.std(tokens)
    mean_times, std_times = np.mean(times), np.std(times)

    print(f"--- '{model_name}' INFO ---")
    print(f"CORRECT FILLED:   mean={mean_good:6.6},\tstd={std_good:6.6}")
    print(f"INCORRECT FILLED: mean={mean_bad:6.6},\tstd={std_bad:6.6}")
    print(f"TOKENS:         mean={mean_tokens:6.6},\tstd={std_tokens:6.6}")
    print(f"TIME:           mean={mean_times:6.6},\tstd={std_times:6.6}")
    print()